In [1]:
from utils import *
import wandb
from datetime import datetime
# from apollo_torch import APOLLOAdamW

In [2]:
config = Config().load(os.path.join("configs", "vit.json"))

In [3]:
def itertoolsBetter(dataIter):
    while True:
        for batch in dataIter:
            yield batch

In [4]:
import signal

def checkpointModel(stamp, config, model):
    # Block SIGINT for the duration of the save
    original = signal.signal(signal.SIGINT, signal.SIG_IGN)
    try:
        latestPath = os.path.join("checkpoints", "pretrain", "latest")
        if not os.path.exists(latestPath):
            os.mkdir(latestPath)

        timePath = os.path.join("checkpoints", "pretrain", stamp)
        if not os.path.exists(timePath):
            os.mkdir(timePath)

        for path in [latestPath, timePath]:
            tmp = os.path.join(path, "checkpoint.tmp")
            dst = os.path.join(path, "checkpoint.pt")
            torch.save(model.state_dict(), tmp)
            os.replace(tmp, dst)
            config.save(os.path.join(path, "config.json"))
    finally:
        signal.signal(signal.SIGINT, original)


def trainModel(config, modelClass, datasetClass, resumePath=None, resumeNumber=None, resumeID=None):
    dataset = datasetClass(config.dataset)
    model = modelClass(config.model)

    if resumePath is not None:
        model.load_state_dict(torch.load(resumePath, weights_only=True))

    model = model.to(device)

    print(f"Model has {sum([p.numel() for p in model.parameters()])} parameters")

    optimizer = torch.optim.Adam(model.parameters(), lr=config.learningRate)

    trainQueue = MoCoQueue(dim=config.model.embedDim, size=config.queueSize)
    testQueue = MoCoQueue(dim=config.model.embedDim, size=config.queueSize)
    momentumModel = MomentumEncoder(model, momentum=config.momentum if "momentum" in config else 0.999)
    objective = EmbeddingLoss(temperature=0.04, alpha=0.1)

    train, test = datasetClass.split(dataset, config)
    train = DataLoader(train, batch_size=config.batchSize, shuffle=True, collate_fn=dataset.collate)
    test = DataLoader(test, batch_size=config.batchSize, shuffle=True, collate_fn=dataset.collate)

    testIter = itertoolsBetter(test)

    run = None
    total = 0 if resumeNumber is None else resumeNumber

    stamp = datetime.now().strftime("%Y-%m-%d %H-%M")

    try:
        for epoch in range(config.epochs):
            progress = 0
            for inputsX, inputsY, info, ids in train:
                model.train()
                optimizer.zero_grad()

                _, outputsX = model(inputsX.to(device))
                _, outputsY = model(inputsY.to(device))
                loss = objective(outputsX, outputsY, ids.to(device), queue=trainQueue, enqueue=False)

                trainLoss = loss["total"].detach().item()

                loss["total"].backward()
                optimizer.step()
                momentumModel.update(model)

                # Queue keys come from the momentum encoder so they stay consistent
                with torch.no_grad():
                    _, keysX = momentumModel(inputsX.to(device))
                    _, keysY = momentumModel(inputsY.to(device))
                trainQueue.enqueue(keysX, keysY, ids.to(device))

                with torch.no_grad():
                    model.eval()
                    inputsX1, inputsY1, info1, ids1 = next(testIter)

                    _, outputsX1 = model(inputsX1.to(device))
                    _, outputsY1 = model(inputsY1.to(device))
                    loss1 = objective(outputsX1, outputsY1, ids1.to(device), queue=testQueue, enqueue=False)

                    with torch.no_grad():
                        _, keysX1 = momentumModel(inputsX1.to(device))
                        _, keysY1 = momentumModel(inputsY1.to(device))
                    testQueue.enqueue(keysX1, keysY1, ids1.to(device))

                    testLoss = loss1["total"].detach().item()

                if run is None:
                    if resumeID is None:
                        run = wandb.init(entity="dylanberndt123-missouri-state-university", project="Briefcase", config=config.serialize())
                    else:
                        run = wandb.init(entity="dylanberndt123-missouri-state-university", project="Briefcase", config=config.serialize(), id=resumeID, resume="allow")
                
                run.log({"Train Loss": trainLoss, 
                         "Train InfoNCE": loss["info"].detach().item(),
                         "Train SigREG": loss["sig"].detach().item(),
                         "Test Loss": testLoss,
                         "Test InfoNCE": loss1["info"].detach().item(),
                         "Test SigREG": loss1["sig"].detach().item(),
                         "Train Perplexity": torch.exp(loss["info"].detach()).item(),
                         "Test Perplexity": torch.exp(loss1["info"].detach()).item(),
                         "Train Recall@1": recallAtK(outputsX, outputsY, k=1, families=ids.to(device), queue=trainQueue),
                         "Test Recall@1": recallAtK(outputsX1, outputsY1, k=1, families=ids1.to(device), queue=testQueue),
                         "Train Recall@5": recallAtK(outputsX, outputsY, k=5, families=ids.to(device), queue=trainQueue),
                         "Test Recall@5": recallAtK(outputsX1, outputsY1, k=5, families=ids1.to(device), queue=testQueue),
                         "Train Recall@10": recallAtK(outputsX, outputsY, k=10, families=ids.to(device), queue=trainQueue),
                         "Test Recall@10": recallAtK(outputsX1, outputsY1, k=10, families=ids1.to(device), queue=testQueue)
                         }, step=total)

                progress += 1
                total += 1

                print(f"\r{epoch + 1} | {progress}/{len(train)} | {(progress / len(train)) * 100:.3f}% |  Train Loss: {trainLoss:.2f} | Test Loss: {testLoss:.2f}", end="")

                if total % 1000 == 0:
                    checkpointModel(stamp, config, model)

    except KeyboardInterrupt:
        checkpointModel(stamp, config, model)

In [ ]:
trainModel(config, ViT, PairedImageData)


Loading MyFonts images from dataset ====================


Fonts serialized: 1855/3794google\fonts\ofl\kumarone\KumarOne-Regular.ttf execution context too long
Fonts serialized: 2335/3794google\fonts\ofl\notocoloremojicompattest\NotoColorEmojiCompatTest-Regular.ttf invalid pixel size
Fonts serialized: 3735/3794google\fonts\ofl\zcoolxiaowei\ZCOOLXiaoWei-Regular.ttf [Errno 22] Invalid argument: 'google\\bitmaps\\????? ?? al.bmp'
Fonts serialized: 3794/3794


Fonts serialized: 36/18623dafont\fonts\135atom_sans.ttf [Errno 2] No such file or directory: 'dafont\\bitmaps\\13/5Atom Sans Regular al.bmp'
Fonts serialized: 79/18623dafont\fonts\50fifty.ttf [Errno 2] No such file or directory: 'dafont\\bitmaps\\50/fifty Regular al.bmp'
Fonts serialized: 353/18623dafont\fonts\adrenaline_zero.ttf [Errno 22] Invalid argument: 'dafont\\bitmaps\\Adrenaline : Zero Adrenaline : Zero al.bmp'
Fonts serialized: 410/18623dafont\fonts\aggstock.ttf [Errno 22] Invalid argument: 'dafont\\bitmaps\\?ggstock Regula

cmap subtable is reported as having zero length: platformID 1, platEncID 0, format 0 offset 20. Skipping table.


Fonts serialized: 3910/18623

cmap subtable is reported as having zero length: platformID 1, platEncID 0, format 0 offset 20. Skipping table.


dafont\fonts\doilie.ttf [Errno 22] Invalid argument: 'dafont\\bitmaps\\do i lie? liars al.bmp'
Fonts serialized: 3998/18623dafont\fonts\dragonforce.ttf [Errno 22] Invalid argument: 'dafont\\bitmaps\\?DragonForcE? Regular al.bmp'
Fonts serialized: 4001/18623dafont\fonts\dragos.ttf [Errno 22] Invalid argument: 'dafont\\bitmaps\\Dragos Brush ? al.bmp'
Fonts serialized: 4307/18623dafont\fonts\elegantech-.ttf [Errno 22] Invalid argument: 'dafont\\bitmaps\\EleganTech? Regular al.bmp'
Fonts serialized: 4346/18623dafont\fonts\elsö.ttf [Errno 22] Invalid argument: 'dafont\\bitmaps\\Els? Regular al.bmp'
Fonts serialized: 4860/18623

...

Fonts serialized: 5833/18623dafont\fonts\gomarice_whats_love_oshare.ttf [Errno 22] Invalid argument: "dafont\\bitmaps\\What's Love? Oshare Regular al.bmp"
Fonts serialized: 6011/18623dafont\fonts\groovenext-.ttf [Errno 22] Invalid argument: 'dafont\\bitmaps\\GrooveNext? Regular al.bmp'
Fonts serialized: 6553/18623dafont\fonts\humbug.ttf corrupt cmap table format 0 (data length: 534, header length: 598)
Fonts serialized: 6561/18623

3 extra bytes in post.stringData array


Fonts serialized: 7066/18623dafont\fonts\jerónimo_cartoon.ttf [Errno 22] Invalid argument: 'dafont\\bitmaps\\Jer?nimo cartoon Regular al.bmp'
Fonts serialized: 7162/18623dafont\fonts\julia_black_extended.ttf Not a TrueType or OpenType font (bad sfntVersion)
Fonts serialized: 7317/18623dafont\fonts\kaylafiz.ttf division by zero
Fonts serialized: 7979/18623dafont\fonts\lavoshandy_99.ttf [Errno 22] Invalid argument: 'dafont\\bitmaps\\LavosHandy? Regular al.bmp'
Fonts serialized: 8089/18623dafont\fonts\lettif__.ttf [Errno 22] Invalid argument: 'dafont\\bitmaps\\Letters II "Fenotype" al.bmp'
Fonts serialized: 8161/18623dafont\fonts\like_giselle.ttf [Errno 22] Invalid argument: 'dafont\\bitmaps\\Like Giselle? Regular al.bmp'
Fonts serialized: 8285/18623dafont\fonts\lomonesia-_dingbats.ttf [Errno 22] Invalid argument: 'dafont\\bitmaps\\Lomonesia? Dingbats Regular al.bmp'
Fonts serialized: 8322/18623dafont\fonts\louis_george_cafe.ttf [Errno 22] Invalid argument: 'dafont\\bitmaps\\Louis George 

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\dylan\_netrc.
wandb: Currently logged in as: dylanberndt123 (dylanberndt123-missouri-state-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


1 | 23673/915733 | 2.585% |  Train Loss: 5.41 | Test Loss: 8.07